In [1]:
from unsloth import FastLanguageModel
import torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/mxs220189/miniconda3/envs/pyllmpatch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
_MODEL = None
_TOKENIZER = None

def load_model_once(
    *,
    model_path: str,
    device_map: str = "auto",
    max_tokens: int
):
    """
    Load the merged model and tokenizer exactly once.
    """
    global _MODEL, _TOKENIZER

    if _MODEL is not None and _TOKENIZER is not None:
        return _MODEL, _TOKENIZER


    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=max_tokens + 8192,
        dtype=None,
        load_in_4bit=True,
        device_map=device_map,
    )

    # model.eval()
    FastLanguageModel.for_inference(model)

    _MODEL = model
    _TOKENIZER = tokenizer

    return _MODEL, _TOKENIZER


In [3]:
messages = [
        {
            "role": "system",
            "content": "You are a python programmer. Answer the question as best as you can. If you don't know the answer, say you don't know.",
        },
        {
            "role": "user",
            "content": "Write a python function that takes in a list of numbers and returns the sum of the numbers.",
        },
    ]

In [4]:
model, tokenizer = load_model_once(model_path="Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled", max_tokens=16384)

==((====))==  Unsloth 2026.3.11: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.69 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Fetching 11 files: 100%|██████████| 11/11 [06:49<00:00, 37.27s/it]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights:   1%|          | 6/1184 [00:31<1:17:03,  3.92s/it] /home/mxs220189/miniconda3/envs/pyllmpatch/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 1184/1184 [03:51<00:00,  5.11it/s]


In [5]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",).to(model.device)
outputs = model.generate(**inputs, max_new_tokens=16384)
tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

TypeError: string indices must be integers